In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # because you are inside Notebooks/
from src.config_loader import load_config

config_path = PROJECT_ROOT / "configs" / "battery_config.yaml"
battery, markets = load_config(config_path)


print("=== BatterySpec (loaded from YAML) ===")
print(battery)

print("\n=== Market configs ===")
for k, v in markets.items():
    print(k, "->", v)

# 2) Discretize for each market (this is the key step)
da_cfg = markets["da"]
ip_cfg = markets["ip"]
print('da_cfg:', da_cfg)
print('ip_cfg:', ip_cfg)

b_da = battery.discretize(da_cfg.dt_minutes)
b_ip = battery.discretize(ip_cfg.dt_minutes)

print("\n=== BatteryDiscrete for DA (hourly) ===")
print(b_da)

print("\n=== BatteryDiscrete for IP (15-min) ===")
print(b_ip)

# 3) Show the main difference explicitly
print("\n=== Key per-step limits ===")
print(f"DA dt_hours = {b_da.dt_hours}, max charge per step = {b_da.e_charge_step_max_kwh:.2f} kWh")
print(f"IP dt_hours = {b_ip.dt_hours}, max charge per step = {b_ip.e_charge_step_max_kwh:.2f} kWh")

# 4) Tiny example: one-step SOC update (no solver)
# Decision variables in an optimization are usually:
#   e_ch[t]   = energy charged in step t (kWh), bounded by e_charge_step_max_kwh
#   e_dis[t]  = energy discharged in step t (kWh), bounded by e_discharge_step_max_kwh
# SOC dynamics (energy form) with efficiencies:
#   e_next = e_now + eta_charge * e_ch - (1/eta_discharge) * e_dis
#
# Let's "try" a charge action and see the updated energy.
e_now = b_ip.e_init_kwh
e_ch = 50.0   # kWh charged in a 15-min step (must be <= 125 if p_charge=500kW)
e_dis = 0.0

e_next = e_now + battery.eta_charge * e_ch - (1.0 / battery.eta_discharge) * e_dis

print("\n=== One-step example on IP grid ===")
print(f"e_now  = {e_now:.2f} kWh")
print(f"e_ch   = {e_ch:.2f} kWh (limit {b_ip.e_charge_step_max_kwh:.2f})")
print(f"e_next = {e_next:.2f} kWh")
print(f"bounds = [{b_ip.e_min_kwh:.2f}, {b_ip.e_max_kwh:.2f}] kWh")
print("feasible:", (b_ip.e_min_kwh <= e_next <= b_ip.e_max_kwh))


=== BatterySpec (loaded from YAML) ===
BatterySpec(name='BESS_1', energy_kwh=1000.0, p_charge_kw_max=500.0, p_discharge_kw_max=500.0, eta_charge=0.96, eta_discharge=0.96, soc_min=0.1, soc_max=0.9, soc_init=0.5, soc_target=None)

=== Market configs ===
da -> MarketConfig(name='da', dt_minutes=60, horizon_steps=24, grid_fee_eur_per_mwh=0.0, allow_simultaneous_charge_discharge=False)
ip -> MarketConfig(name='ip', dt_minutes=15, horizon_steps=96, grid_fee_eur_per_mwh=0.0, allow_simultaneous_charge_discharge=False)
da_cfg: MarketConfig(name='da', dt_minutes=60, horizon_steps=24, grid_fee_eur_per_mwh=0.0, allow_simultaneous_charge_discharge=False)
ip_cfg: MarketConfig(name='ip', dt_minutes=15, horizon_steps=96, grid_fee_eur_per_mwh=0.0, allow_simultaneous_charge_discharge=False)

=== BatteryDiscrete for DA (hourly) ===
BatteryDiscrete(spec=BatterySpec(name='BESS_1', energy_kwh=1000.0, p_charge_kw_max=500.0, p_discharge_kw_max=500.0, eta_charge=0.96, eta_discharge=0.96, soc_min=0.1, soc_max=0